In [1]:
import re
import pandas as pd
from rapidfuzz.distance import JaroWinkler
import itertools

def std_name(name):
    prefixes = r'^(?:(?:Adv|Dr|Mr|Mrs|Ms|Shri|Smt|Prof)\.?\s+)+'
    name = re.sub(prefixes, "", name, flags=re.IGNORECASE)

    lineage_field = r'\s+(?:S/o|D/o|W/o|C/o|H/o|@)\.?\s+.*$'
    name = re.sub(lineage_field, "", name, flags=re.IGNORECASE)

    name = name.split(",", 1)[0]
    name = name.split("(", 1)[0]
    name = name.replace(".", " ")

    words = name.split()
    words = [word.lower() for word in words]
    return " ".join(words)

def names_agree(a, b, threshold=0.80):
    words_a = a.split()
    words_b = b.split()
    if len(words_a) <= len(words_b):
        short, long = words_a, words_b
    else:
        short, long = words_b, words_a

    def words_match(w1, w2):
        if len(w1) == 1 or len(w2) == 1:
            return w1[0] == w2[0]
        return JaroWinkler.similarity(w1, w2) >= threshold

    for perm in itertools.permutations(long, len(short)):
        if all(words_match(s, l) for s, l in zip(short, perm)):
            return True
    return False


def match_candidates(
    election_df,
    winners_df,
    election_name_col="Candidate",
    winner_name_col="Winner Candidate",
):
    election_df = election_df.copy()
    winners_df = winners_df.copy()

    election_df["_std_name"] = election_df[election_name_col].apply(std_name)
    winners_df["_std_name"] = winners_df[winner_name_col].apply(std_name)

    matches = []

    # Outer loop: winners
    for i, winner_row in winners_df.iterrows():
        name1 = winner_row["_std_name"]

        if not name1:
            continue

        best_match = None
        best_score = 0

        # Inner loop: election candidates
        for j, election_row in election_df.iterrows():
            name2 = election_row["_std_name"]

            if not name2:
                continue

            if names_agree(name1, name2):
                score = JaroWinkler.similarity(name1, name2)

                if score > best_score:
                    best_score = score
                    best_match = {
                        "winner_candidate": winner_row[winner_name_col],
                        "winner_of_constituency": winner_row["Constituency"],
                        "election_candidate": election_row[election_name_col],
                        "similarity": score,
                        "party": winner_row.get("Winner Party", None),
                        "winner_index": i,
                        "election_index": j
                    }

        if best_match is not None:
            matches.append(best_match)

    return pd.DataFrame(matches)

In [2]:
df = pd.read_csv("loksabha_winners_2014_2019_2024_normalized.csv")

In [3]:
andhra_14_winners = df[(df["State"]=="Andhra Pradesh")&(df["Election Year"]==2014)]
andhra_14 = pd.read_parquet("cleaned_data/imputed_parquet/andhra_2014_imputed.parquet")
matching_andhra_14 = match_candidates(andhra_14, andhra_14_winners)
matching_andhra_14

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Kothapalli Geetha,Araku,Kothapalli Geetha,1.000000,YSRCP,0,146
1,Ram Mohan Naidu Kinjarapu,Srikakulam,Rammohannaidu Kinjarapu,0.930609,TDP,1,234
2,Ashok Gajapathi Raju,Vizianagaram,Ashok Gajapathi Raju Pusapati,0.937931,TDP,2,15
3,Kambhampati Hari Babu,Visakhapatnam,Kambhampati Hari Babu,1.000000,BJP,3,69
4,Muttamsetti Srinivasa Rao,Anakapalle,Muttamsetti Srinivasa Rao,1.000000,TDP,4,185
5,Thota Narasimham,Kakinada,Thota Venkata Narasimham,0.895833,TDP,5,282
6,Pandula Ravindra Babu,Amalapuram,Pandula Ravindra Babu,1.000000,TDP,6,209
7,Murali Mohan,Rajahmundry,Murali Mohan Maganti,0.920000,TDP,7,183
8,Gokaraju Ganga Raju,Narasapuram,Gokaraju Ganga Raju,1.000000,BJP,8,87
9,Maganti Venkateswara Rao,Eluru,Venkateswara Rao Maganti,0.789683,TDP,9,309


**Manual Checking**

In [4]:
for i in list(andhra_14["Candidate"]):
    if names_agree(std_name(i), "y s avinash reddy"):
        print(i)

Avinash Reddy Yeduguri Sandinti
Shanthi Reddy
Yeluri Srilatha


In [5]:
matching_andhra_14.loc[matching_andhra_14["winner_candidate"] == "Y. S. Avinash Reddy", "election_candidate"] = "Avinash Reddy Yeduguri Sandinti"
matching_andhra_14

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Kothapalli Geetha,Araku,Kothapalli Geetha,1.000000,YSRCP,0,146
1,Ram Mohan Naidu Kinjarapu,Srikakulam,Rammohannaidu Kinjarapu,0.930609,TDP,1,234
2,Ashok Gajapathi Raju,Vizianagaram,Ashok Gajapathi Raju Pusapati,0.937931,TDP,2,15
3,Kambhampati Hari Babu,Visakhapatnam,Kambhampati Hari Babu,1.000000,BJP,3,69
4,Muttamsetti Srinivasa Rao,Anakapalle,Muttamsetti Srinivasa Rao,1.000000,TDP,4,185
5,Thota Narasimham,Kakinada,Thota Venkata Narasimham,0.895833,TDP,5,282
6,Pandula Ravindra Babu,Amalapuram,Pandula Ravindra Babu,1.000000,TDP,6,209
7,Murali Mohan,Rajahmundry,Murali Mohan Maganti,0.920000,TDP,7,183
8,Gokaraju Ganga Raju,Narasapuram,Gokaraju Ganga Raju,1.000000,BJP,8,87
9,Maganti Venkateswara Rao,Eluru,Venkateswara Rao Maganti,0.789683,TDP,9,309


In [6]:
andhra_19_winners = df[(df["State"]=="Andhra Pradesh")&(df["Election Year"]==2019)]
andhra_19 = pd.read_parquet("cleaned_data/imputed_parquet/andhra_2019_imputed.parquet")
matching_andhra_19 = match_candidates(andhra_19, andhra_19_winners)
matching_andhra_19

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Goddeti Madhavi,Araku (ST),Goddeti Madhavi,1.000000,YSRCP,91,96
1,Kinjarapu Ram Mohan Naidu,Srikakulam,Rammohannaidu Kinjarapu,0.721678,TDP,92,245
2,Bellana Chandra Sekhar,Vizianagaram,Bellana Chandra Sekhar,1.000000,YSRCP,93,31
3,M. V. V. Satyanarayana,Visakhapatnam,M V V Satyanarayana,1.000000,YSRCP,94,162
4,Beesetti Venkata Satyavathi,Anakapalle,Dr Beesetti Venkata Satyavathi,1.000000,YSRCP,95,77
5,Vanga Geetha,Kakinada,Vanga Geetha Viswanatham,0.900000,YSRCP,96,295
6,Chinta Anuradha,Amalapuram (SC),Chinta Anuradha,1.000000,YSRCP,97,54
7,Margani Bharat,Rajahmundry,Margani Bharat Ram,0.955556,YSRCP,98,177
8,Raghu Rama Krishna Raju,Narasapuram,Kanumuru Raghu Rama Krishna Raju,0.761322,YSRCP,99,140
9,Kotagiri Sridhar,Eluru,K T Sreedhar,0.622685,YSRCP,100,131


In [7]:
andhra_24_winners = df[(df["State"]=="Andhra Pradesh")&(df["Election Year"]==2024)]
andhra_24 = pd.read_parquet("cleaned_data/imputed_parquet/andhra_2024_imputed.parquet")
matching_andhra_24 = match_candidates(andhra_24, andhra_24_winners)
matching_andhra_24

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Gumma Thanuja Rani,Araku (ST),Gumma Thanuja Rani,1.000000,YSRCP,182,175
1,K. Ram Mohan Naidu,Srikakulam,Kinjarapu Rammohan Naidu,0.751103,TDP,183,222
2,Appala Naidu Kalisetti,Vizianagaram,Appalanaidu Kalisetti,0.962338,TDP,184,32
3,C. M. Ramesh,Anakapalle,C M Ramesh,1.000000,BJP,186,95
4,Tangella Uday Srinivas,Kakinada,Tangella Uday Srinivas,1.000000,Janasena Party,187,391
5,Ganti Harish Madhur,Amalapuram (SC),G M Harish,0.657018,TDP,188,149
6,Daggubati Purandeswari,Rajahmundry,Daggubati Purandheshwari,0.983333,BJP,189,115
7,Bhupathi Srinivasa Varma,Narasapuram,Bhupathi Raju Srinivasa Varma,0.907184,BJP,190,70
8,Putta Mahesh Kumar,Eluru,Putta Mahesh Kumar,1.000000,TDP,191,336
9,Vallabhaneni Balashowry,Machilipatnam,Balashowry Vallabhaneni,0.620083,Janasena Party,192,52


In [8]:
bihar_24_winners = df[(df["State"] == "Bihar") & (df["Election Year"] == 2024)]
bihar_24 = pd.read_parquet("cleaned_data/imputed_parquet/bihar_2024_imputed.parquet")
matching_bihar_24 = match_candidates(bihar_24, bihar_24_winners)
matching_bihar_24

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Sunil Kumar,Valmiki Nagar,Sunil Kumar,1.000000,JD(U),207,444
1,Sanjay Jaiswal,Paschim Champaran,Dr Sanjay Jaiswal,1.000000,BJP,208,128
2,Radha Mohan Singh,Purvi Champaran,Radha Mohan Singh,1.000000,BJP,209,311
3,Lovely Anand,Sheohar,Lovely Anand,1.000000,JD(U),210,201
4,Devesh Chandra Thakur,Sitamarhi,Devesh Chandra Thakur,1.000000,JD(U),211,104
5,Ashok Kumar Yadav,Madhubani,Ashok Kumar Yadav,1.000000,BJP,212,57
6,Ramprit Mandal,Jhanjharpur,Ramprit Mandal,1.000000,JD(U),213,355
7,Dileshwar Kamait,Supaul,Dileshwar Kamait,1.000000,JD(U),214,112
8,Pradeep Kumar Singh,Araria,Pradeep Kumar Singh,1.000000,BJP,215,294
9,Mohammad Jawed,Kishanganj,Mohammad Jawed,1.000000,INC,216,234


**Manual Checking**

In [9]:
for i in list(bihar_24["Candidate"]):
    if names_agree(std_name(i), "lalan singh"):
        print(i)

Lal Badsah Singh
Rajiv Ranjan Singh Alias Lalan Singh


In [10]:
matching_bihar_24.loc[matching_bihar_24["winner_candidate"] == "Lalan Singh", "election_candidate"] = "Rajiv Ranjan Singh Alias Lalan Singh"
matching_bihar_24

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Sunil Kumar,Valmiki Nagar,Sunil Kumar,1.000000,JD(U),207,444
1,Sanjay Jaiswal,Paschim Champaran,Dr Sanjay Jaiswal,1.000000,BJP,208,128
2,Radha Mohan Singh,Purvi Champaran,Radha Mohan Singh,1.000000,BJP,209,311
3,Lovely Anand,Sheohar,Lovely Anand,1.000000,JD(U),210,201
4,Devesh Chandra Thakur,Sitamarhi,Devesh Chandra Thakur,1.000000,JD(U),211,104
5,Ashok Kumar Yadav,Madhubani,Ashok Kumar Yadav,1.000000,BJP,212,57
6,Ramprit Mandal,Jhanjharpur,Ramprit Mandal,1.000000,JD(U),213,355
7,Dileshwar Kamait,Supaul,Dileshwar Kamait,1.000000,JD(U),214,112
8,Pradeep Kumar Singh,Araria,Pradeep Kumar Singh,1.000000,BJP,215,294
9,Mohammad Jawed,Kishanganj,Mohammad Jawed,1.000000,INC,216,234


In [11]:
for i in list(bihar_24["Candidate"]):
    if names_agree(std_name(i), "abhay kushwaha"):
        print(i)

A K Singh


In [12]:
bihar_19_winners = df[(df["State"] == "Bihar") & (df["Election Year"] == 2019)]
bihar_19 = pd.read_parquet("cleaned_data/imputed_parquet/bihar_2019_imputed.parquet")
matching_bihar_19 = match_candidates(bihar_19, bihar_19_winners)
matching_bihar_19


,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Baidyanath Prasad Mahto,Valmiki Nagar,Baidyanath Prasad Mahto,1.000000,JD(U),116,80
1,Sanjay Jaiswal,Paschim Champaran,Sanjay Jaiswal,1.000000,BJP,117,155
2,Radha Mohan Singh,Purvi Champaran,Radha Mohan Singh,1.000000,BJP,118,351
3,Rama Devi,Sheohar,Rama Devi,1.000000,BJP,119,410
4,Sunil Kumar Pintu,Sitamarhi,Sunil Kumar,0.929412,JD(U),120,542
5,Ashok Kumar Yadav,Madhubani,Ashok Kumar Yadav,1.000000,BJP,121,74
6,Ramprit Mandal,Jhanjharpur,Ramprit Mandal,1.000000,JD(U),122,428
7,Dileshwar Kamait,Supaul,Dileshwar Kamait,1.000000,JD(U),123,136
8,Pradeep Kumar Singh,Araria,Pradeep Kumar Singh,1.000000,BJP,124,331
9,Mohammad Jawed,Kishanganj,Mohammad Jawed,1.000000,INC,125,153


In [13]:
bihar_14_winners = df[(df["State"] == "Bihar") & (df["Election Year"] == 2014)]
bihar_14 = pd.read_parquet("cleaned_data/imputed_parquet/bihar_2014_imputed.parquet")
matching_bihar_14 = match_candidates(bihar_14, bihar_14_winners)
matching_bihar_14

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Satish Chandra Dubey,Valmiki Nagar,Satish Dubey,0.920000,BJP,25,485
1,Sanjay Jaiswal,Paschim Champaran,Sanjay Jayaswal,0.943663,BJP,26,134
2,Radha Mohan Singh,Purvi Champaran,Radha Mohan Singh,1.000000,BJP,27,381
3,Rama Devi,Sheohar,Rama Devi,1.000000,BJP,28,430
4,Ram Kumar Sharma,Sitamarhi,Ram Kumar Sharma,1.000000,Rashtriya Lok Samta Party,29,423
5,Hukmdev Narayan Yadav,Madhubani,Hukum Deo Narayan Yadav,0.900121,BJP,30,165
6,Birendra Chaudhary,Jhanjharpur,Bijendra Chaudhary,0.938998,BJP,31,75
7,Ranjeet Ranjan,Supaul,Ranjeet Ranjan,1.000000,INC,32,450
8,Taslimuddin,Araria,Taslim Uddin,0.983333,RJD,33,565
9,Tariq Anwar,Katihar,Md Tariq Anwar,0.837662,NCP,35,270


In [14]:
gujarat_14_winners = df[(df["State"] == "Gujarat") & (df["Election Year"] == 2014)]
gujarat_14 = pd.read_parquet("cleaned_data/imputed_parquet/gujarat_2014_imputed.parquet")
matching_gujarat_14 = match_candidates(gujarat_14, gujarat_14_winners)
matching_gujarat_14

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Vinodbhai Chavda,Kachchh,Chavda Vinod Lakhmashi,0.694264,BJP,65,43
1,Haribhai Chaudhary,Banaskantha,Harilal Ranchodbhai Chauhan,0.844227,BJP,66,81
2,Liladhar Vaghela,Patan,Vaghela Liladharbhai Khodaji,0.711310,BJP,67,313
3,Jayshreeben Patel,Mahesana,Patel Jayshreeben Kanubhai,0.747360,BJP,68,193
4,Dipsinh Rathod,Sabarkantha,Rathod Dipsinh Shankarsinh,0.679487,BJP,69,235
5,L. K. Advani,Gandhinagar,Lal Krishna Advani,0.746667,BJP,70,117
6,Paresh Rawal,Ahmedabad East,Paresh Dahyalal Rawal,0.880952,BJP,71,172
7,Kirit Solanki,Ahmedabad West,Solanki Kiritbhai Premjibhai,0.667582,BJP,72,66
8,Devajibhai Fatepara,Surendranagar,Fatepara Devajibhai Govindbhai,0.719883,BJP,73,68
9,Mohan Kundariya,Rajkot,Mohanbhai Kalyanjibhai Kundariya,0.641667,BJP,74,148


In [15]:
gujarat_19_winners = df[(df["State"] == "Gujarat") & (df["Election Year"] == 2019)]
gujarat_19 = pd.read_parquet("cleaned_data/imputed_parquet/gujarat_2019_imputed.parquet")
matching_gujarat_19 = match_candidates(gujarat_19, gujarat_19_winners)
matching_gujarat_19

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Vinodbhai Chavda,Kachchh,Chavda Vinod Lakhamshi,0.694264,BJP,156,69
1,Parbatbhai Patel,Banaskantha,Parbatbhai Patel,1.000000,BJP,157,206
2,Shardaben Patel,Mahesana,Shardaben Anilbhai Patel,0.898333,BJP,159,299
3,Dipsinh Rathod,Sabarkantha,Rathod Dipsinh Shankarsinh,0.679487,BJP,160,269
4,Amit Shah,Gandhinagar,Amit Shah,1.000000,BJP,161,9
5,Hasmukh Patel,Ahmedabad East,Patel Hasmukhbhai Somabhai,0.705128,BJP,162,222
6,Kirit Solanki,Ahmedabad West,Solanki Kiritbhai,0.679556,BJP,163,98
7,Mahendra Munjapara,Surendranagar,Mahendrabhai Munjpara,0.938095,BJP,164,168
8,Mohan Kundariya,Rajkot,Kundaria Mohanbhai,0.629966,BJP,165,159
9,Rameshbhai Dhaduk,Porbandar,Rameshbhai Dhaduk,1.000000,BJP,166,258


In [16]:
for i in list(gujarat_19["Candidate"]):
    if names_agree(std_name(i), std_name("Rajesh Chudasama")):
        print(i)

Chudasama Rajeshbhai Naranbhai
Rakeshbhai


In [17]:
matching_gujarat_19.loc[matching_gujarat_19["winner_candidate"] == "Rajesh Chudasama", "election_candidate"] = "Chudasama Rajeshbhai Naranbhai"
matching_gujarat_19

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Vinodbhai Chavda,Kachchh,Chavda Vinod Lakhamshi,0.694264,BJP,156,69
1,Parbatbhai Patel,Banaskantha,Parbatbhai Patel,1.000000,BJP,157,206
2,Shardaben Patel,Mahesana,Shardaben Anilbhai Patel,0.898333,BJP,159,299
3,Dipsinh Rathod,Sabarkantha,Rathod Dipsinh Shankarsinh,0.679487,BJP,160,269
4,Amit Shah,Gandhinagar,Amit Shah,1.000000,BJP,161,9
5,Hasmukh Patel,Ahmedabad East,Patel Hasmukhbhai Somabhai,0.705128,BJP,162,222
6,Kirit Solanki,Ahmedabad West,Solanki Kiritbhai,0.679556,BJP,163,98
7,Mahendra Munjapara,Surendranagar,Mahendrabhai Munjpara,0.938095,BJP,164,168
8,Mohan Kundariya,Rajkot,Kundaria Mohanbhai,0.629966,BJP,165,159
9,Rameshbhai Dhaduk,Porbandar,Rameshbhai Dhaduk,1.000000,BJP,166,258


In [18]:
for i in list(gujarat_19["Candidate"]):
    if names_agree(std_name(i), std_name("C. R. Patil")):
        print(i)

Chandansinh
Chandraben
Patil Chandrakant
Rakeshbhai
Reshma Patel
Rita Captain


In [19]:
matching_gujarat_19.loc[matching_gujarat_19["winner_candidate"] == "C. R. Patil", "election_candidate"] = "Patil Chandrakant"
matching_gujarat_19

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Vinodbhai Chavda,Kachchh,Chavda Vinod Lakhamshi,0.694264,BJP,156,69
1,Parbatbhai Patel,Banaskantha,Parbatbhai Patel,1.000000,BJP,157,206
2,Shardaben Patel,Mahesana,Shardaben Anilbhai Patel,0.898333,BJP,159,299
3,Dipsinh Rathod,Sabarkantha,Rathod Dipsinh Shankarsinh,0.679487,BJP,160,269
4,Amit Shah,Gandhinagar,Amit Shah,1.000000,BJP,161,9
5,Hasmukh Patel,Ahmedabad East,Patel Hasmukhbhai Somabhai,0.705128,BJP,162,222
6,Kirit Solanki,Ahmedabad West,Solanki Kiritbhai,0.679556,BJP,163,98
7,Mahendra Munjapara,Surendranagar,Mahendrabhai Munjpara,0.938095,BJP,164,168
8,Mohan Kundariya,Rajkot,Kundaria Mohanbhai,0.629966,BJP,165,159
9,Rameshbhai Dhaduk,Porbandar,Rameshbhai Dhaduk,1.000000,BJP,166,258


In [20]:
gujarat_24_winners = df[(df["State"] == "Gujarat") & (df["Election Year"] == 2024)]
gujarat_24 = pd.read_parquet("cleaned_data/imputed_parquet/gujarat_2024_imputed.parquet")
matching_gujarat_24 = match_candidates(gujarat_24, gujarat_24_winners)
matching_gujarat_24

,winner_candidate,winner_of_constituency,election_candidate,similarity,party,winner_index,election_index
0,Vinodbhai Chavda,Kachchh,Chavda Vinod Lakhamshi,0.694264,BJP,247,58
1,Geni Thakor,Banaskantha,Geniben Nagaji Thakor,0.610390,INC,248,88
2,Bharatsinhji Dabhi,Patan,Dabhi Bharatsinhji Shankarji,0.732804,BJP,249,63
3,Haribhai Patel,Mahesana,Haribhai Patel,1.000000,BJP,250,94
4,Shobhanaben Baraiya,Sabarkantha,Shobhanaben Mahendrasinh Baraiya,0.887171,BJP,251,229
5,Amit Shah,Gandhinagar,Amit Shah,1.000000,BJP,252,8
6,Hasmukh Patel,Ahmedabad East,Hasmukhbhai Patel,0.937557,BJP,253,97
7,Dinesh Makwana,Ahmedabad West,Dineshbhai Makwana,0.898413,BJP,254,79
8,Chandubhai Shihora,Surendranagar,Chandubhai Chhaganbhai Shihora,0.886667,BJP,255,50
9,Parshottam Rupala,Rajkot,Parshottambhai Rupala,0.926611,BJP,256,184


In [21]:
def match_candidates_across_years(
    year1_df,
    year2_df,
    name_col="Candidate"
):
    year1_df = year1_df.copy()
    year2_df = year2_df.copy()

    year1_df["_std_name"] = year1_df[name_col].apply(std_name)
    year2_df["_std_name"] = year2_df[name_col].apply(std_name)

    matches = []

    # For each candidate in year 1, find the best match in year 2
    for i, row1 in year1_df.iterrows():
        name1 = row1["_std_name"]

        if not name1:
            continue

        best_match = None
        best_score = -1

        for j, row2 in year2_df.iterrows():
            name2 = row2["_std_name"]

            if not name2:
                continue

            # Reuse your existing name-matching logic
            if names_agree(name1, name2):
                score = JaroWinkler.similarity(name1, name2)

                if score > best_score:
                    best_score = score
                    best_match = {
                        "candidate_2014": row1[name_col],
                        "candidate_2019": row2[name_col],
                        "similarity": score,
                        "index_2014": i,
                        "index_2019": j
                    }

        if best_match is not None:
            matches.append(best_match)

    return pd.DataFrame(matches)

matching_gujarat_14_to_19 = match_candidates_across_years(gujarat_14, gujarat_19)
matching_gujarat_14_to_19

,candidate_2014,candidate_2019,similarity,index_2014,index_2019
0,Aayar Muljibhai Khanabhai,Amandbhai,0.590688,0,7
1,Arunkumar Sheshram Pathak,Sherkhan Pathan,0.688254,8,306
2,Babubhai Chhaganbhai Talaviya,Babubhai Talaviya,0.881947,10,15
3,Bar Ajamalbhai Karmanbhai,Amandbhai,0.590688,14,7
4,Basirbhai Sulemanbhai Sama,Amandbhai,0.576923,15,7
...,...,...,...,...,...
176,Vaniya Ajmalbhai Ramabhai,Vankar Rameshbhai Valjibhai,0.813461,320,350
177,Vaniya Gangajibhai,Daniya Anirudhbhai Gandabhai,0.700794,321,76
178,Vankar Manilal Bhanabhai,Amandbhai,0.615741,322,7
179,Vasava Kishorbhai Chhotubhai,Chhotubhai Vasava,0.631653,323,70


In [ ]:
def match_repeat_candidates(
    year1_df,
    year2_df,
    name_col="Candidate",
    constituency_col="Constituency",
    threshold=0.88
):
    results = []

    a = year1_df.copy()
    b = year2_df.copy()

    a["_std_name"] = a[name_col].fillna("").apply(std_name)
    b["_std_name"] = b[name_col].fillna("").apply(std_name)

    for i, row1 in a.iterrows():
        name1 = row1["_std_name"]

        if not name1:
            continue

        candidates = []

        for j, row2 in b.iterrows():
            name2 = row2["_std_name"]

            if not name2 or not names_agree(name1, name2):
                continue

            score = JaroWinkler.similarity(name1, name2)

            if score < threshold:
                continue

            same_constituency = (
                constituency_col in a.columns
                and constituency_col in b.columns
                and pd.notna(row1[constituency_col])
                and pd.notna(row2[constituency_col])
                and str(row1[constituency_col]).strip().casefold()
                == str(row2[constituency_col]).strip().casefold()
            )

            candidates.append({
                "candidate_2014": row1[name_col],
                "candidate_2019": row2[name_col],
                "similarity": score,
                "same_constituency": same_constituency,
                "index_2014": i,
                "index_2019": j
            })

        if candidates:
            # Prefer same constituency, then higher name similarity
            best = max(
                candidates,
                key=lambda x: (x["same_constituency"], x["similarity"])
            )
            results.append(best)

    return pd.DataFrame(results)


Found 'Alice' somewhere in the DataFrame


In [35]:
matching_gujarat_14_19 = match_repeat_candidates(
    gujarat_14,
    gujarat_19,
    threshold=0.70
)

# Initially mark everyone as not recontesting
gujarat_19["recontesting"] = False

# Get the indices of candidates matched from 2014
matched_indices = matching_gujarat_14_19["index_2019"].unique()

# Mark matched candidates as recontesting
gujarat_19.loc[matched_indices, "recontesting"] = True
gujarat_19

,Sno.,Candidate,Party,Criminal Cases,Education,Age,Total Assets,Liabilities,recontesting
0,1,Parmar Vijay,BSP,4,12th Pass,48,535000.0,0,True
1,2,A J Patel,INC,0,Graduate Professional,73,698773712.0,96220673,False
2,3,Aamin Mamadbhai Safiya,IND,0,8th Pass,30,46100.0,0,False
3,4,Aantroliya Karabhai,IND,0,8th Pass,40,760000.0,1329600,False
4,5,Abdulkudus Abdulmajid Molapiya,IND,0,Literate,44,756000.0,0,True
...,...,...,...,...,...,...,...,...,...
366,367,Vora Alimahmad Rajabhai,IND,0,Illiterate,67,1970884.0,32000,True
367,368,Yasinali Polra,New All India Congress Party,0,12th Pass,47,3363000.0,2014250,False
368,369,Zala Dalpatsinh Motisinh,IND,0,8th Pass,55,1827000.0,0,False
369,370,Zala Mukeshbhai Bharmalbhai,IND,1,10th Pass,36,10572716.0,1878780,False


In [36]:
matching_gujarat_19_24 = match_repeat_candidates(
    gujarat_19,
    gujarat_24,
    threshold=0.70
)

# Initially mark everyone as not recontesting
gujarat_24["recontesting"] = False

# Get the indices of candidates matched from 2014
matched_indices = matching_gujarat_19_24["index_2019"].unique()

# Mark matched candidates as recontesting
gujarat_24.loc[matched_indices, "recontesting"] = True
gujarat_24

,Sno.,Candidate,Party,Criminal Cases,Education,Age,Total Assets,Liabilities,recontesting
0,1,Aarab Hasam Sumra,IND,0,5th Pass,62,249591,0,False
1,2,Abdulhaq Ismail Nedariya,IND,1,5th Pass,46,4625500,0,False
2,3,Abdulkuddus,IND,0,5th Pass,49,310000,0,True
3,4,Advocate J P Maraviya,INC,0,Graduate Professional,43,10231739,144000,False
4,5,Advocate Navinkumar Shankerbhai Patel,IND,0,Graduate Professional,53,745000,0,False
...,...,...,...,...,...,...,...,...,...
261,262,Yusuf Vali Hasanali,IND,0,8th Pass,49,2731000,600000,False
262,263,Zala Devrajbhai Babubhai,IND,3,8th Pass,34,483000,0,False
263,264,Zala Nayan J,IND,0,8th Pass,39,161640,0,False
264,265,Zala Sanjaykumar Mahobatsinh,IND,0,Graduate,36,400000,0,False


In [32]:
matching_andhra_14_19 = match_repeat_candidates(
    andhra_14,
    andhra_19,
    threshold=0.70
)

# Initially mark everyone as not recontesting
andhra_19["recontesting"] = False

# Get the indices of candidates matched from 2014
matched_indices = matching_andhra_14_19["index_2019"].unique()

# Mark matched candidates as recontesting
andhra_19.loc[matched_indices, "recontesting"] = True
andhra_19

,Sno.,Candidate,Party,Criminal Cases,Education,Age,Total Assets,Liabilities,recontesting
0,1,A Hemanth,IND,0,5th Pass,28.0,5.000000e+04,0,False
1,2,Abdul Waris,SDPI,4,8th Pass,58.0,5.110000e+06,0,False
2,3,Adala Prabhakara Reddy,YSRCP,0,12th Pass,70.0,2.211663e+09,152281255,True
3,4,Adari Anand Kumar,TDP,0,10th Pass,43.0,4.105472e+08,205591400,False
4,5,Adinarayana Inty,BJP,1,Post Graduate,67.0,3.388590e+07,4479155,False
...,...,...,...,...,...,...,...,...,...
309,310,Yanamadala Venkata Suresh,IND,0,Graduate,50.0,1.302000e+06,0,False
310,311,Yarlagadda Rama Mohana Rao,Bahujan Maha Party,0,12th Pass,56.0,1.008600e+07,0,True
311,312,Yeduguri Sandinti Avinash Reddy,YSRCP,4,Post Graduate,34.0,1.869698e+08,22833319,True
312,313,Yella Rao Siyyadula,IND,0,12th Pass,37.0,5.700000e+04,0,True


In [37]:
matching_andhra_19_24 = match_repeat_candidates(
    andhra_19,
    andhra_24,
    threshold=0.70
)

# Initially mark everyone as not recontesting
andhra_24["recontesting"] = False

# Get the indices of candidates matched from 2014
matched_indices = matching_andhra_19_24["index_2019"].unique()

# Mark matched candidates as recontesting
andhra_24.loc[matched_indices, "recontesting"] = True
andhra_24

,Sno.,Candidate,Party,Criminal Cases,Education,Age,Total Assets,Liabilities,recontesting
0,1,L Khaja Hussain,IND,0,Graduate,42,9246142.0,0,False
1,2,A Chinna Moulali,BSP,0,Graduate Professional,43,8508000.0,1300000,False
2,3,A Madhu,Jai Hindustan Party,0,10th Pass,31,54300.0,0,False
3,4,A Purna Chandra,IND,0,10th Pass,37,200000.0,0,False
4,5,A Varaprasad,Jatiya Jana Sena Party,0,12th Pass,51,253427.0,123000,False
...,...,...,...,...,...,...,...,...,...
445,446,Yeddu Lakshminarayana,IND,0,5th Pass,66,830000.0,230000,False
446,447,Yedida Vara Prasad Kumar,Republican Party of India (A),0,12th Pass,55,100000.0,0,False
447,448,Yelampalle Govardhan Reddy,All India Forward Bloc,0,12th Pass,48,3111906.0,0,False
448,449,Yenugupalli Krishna,Jaibhim Rao Bharat Party,4,Graduate,45,6760715.0,3225113,False


In [33]:
matching_bihar_14_19 = match_repeat_candidates(
    bihar_14,
    bihar_19,
    threshold=0.70
)

# Initially mark everyone as not recontesting
bihar_19["recontesting"] = False

# Get the indices of candidates matched from 2014
matched_indices = matching_bihar_14_19["index_2019"].unique()

# Mark matched candidates as recontesting
bihar_19.loc[matched_indices, "recontesting"] = True
bihar_19

,Sno.,Candidate,Party,Criminal Cases,Education,Age,Total Assets,Liabilities,recontesting
0,1,A M Izharul Haque,Sathi Aur Aapka Faisala Party,0,Post Graduate,43,854501.0,0,False
1,2,Aakash Kumar,Rashtriya Lok Samta Party,0,12th Pass,27,25867226.0,0,False
2,3,Abdul Ajij,IND,2,8th Pass,69,435000.0,0,False
3,4,Abdul Bari Siddiqui,RJD,0,12th Pass,67,21127716.0,911152,True
4,5,Abdul Wahid Khan,IND,1,10th Pass,69,1545000.0,119028,False
...,...,...,...,...,...,...,...,...,...
614,615,Vishnu Dev,IND,0,Graduate,59,14210591.0,500000,False
615,616,Vishnu Dev Yadav,BSP,0,Graduate,67,43439000.0,0,False
616,617,Vishnu Priya,Bahujan Mukti Party,0,Graduate,32,8810000.0,0,False
617,618,Wakil Ahmad,Janata Party,0,Graduate,46,6520000.0,0,False


In [38]:
matching_bihar_19_24 = match_repeat_candidates(
    bihar_19,
    bihar_24,
    threshold=0.70
)

# Initially mark everyone as not recontesting
bihar_24["recontesting"] = False

# Get the indices of candidates matched from 2014
matched_indices = matching_bihar_19_24["index_2019"].unique()

# Mark matched candidates as recontesting
bihar_24.loc[matched_indices, "recontesting"] = True
bihar_24

,Sno.,Candidate,Party,Criminal Cases,Education,Age,Total Assets,Liabilities,recontesting
0,1,Abu Bakar Rahmani,Country Citizen Party,0,Post Graduate,35,1358312,0,True
1,2,A K Singh,IND,0,Graduate Professional,69,22062267,0,True
2,3,Aakash Kumar Singh,INC,0,12th Pass,33,65933186,0,True
3,4,Aanand Misra,IND,0,12th Pass,41,5666811,170000,False
4,5,Abdhesh Kumar Roy,CPI,4,12th Pass,72,4554459,0,False
...,...,...,...,...,...,...,...,...,...
491,492,Vishnu Singh,Bharat Jago Janta Party,0,10th Pass,50,3128369,0,False
492,493,Vishwanath Tudu,IND,0,12th Pass,31,100612,0,False
493,494,Vivek Thakur,BJP,0,Post Graduate,54,50566297,10444564,False
494,495,Yog Narayan Singh,IND,0,Literate,49,2695200,0,False


In [ ]:
# def match_repeat_candidates(
#     year1_df,
#     year2_df,
#     name_col="Candidate",
#     party_col="Party",
#     threshold=0.88
# ):
#     a = year1_df.copy()
#     b = year2_df.copy()

#     a["_std_name"] = a[name_col].fillna("").apply(std_name)
#     b["_std_name"] = b[name_col].fillna("").apply(std_name)

#     # Normalize party names
#     a["_std_party"] = (
#         a[party_col].fillna("").astype(str).str.strip().str.casefold()
#     )
#     b["_std_party"] = (
#         b[party_col].fillna("").astype(str).str.strip().str.casefold()
#     )

#     results = []

#     for i, row1 in a.iterrows():
#         name1 = row1["_std_name"]
#         party1 = row1["_std_party"]

#         if not name1 or not party1:
#             continue

#         # Only consider candidates from the SAME party
#         same_party_rows = b[
#             (b["_std_party"] == party1) &
#             (b["_std_party"] != "")
#         ]

#         best_match = None
#         best_score = -1

#         for j, row2 in same_party_rows.iterrows():
#             name2 = row2["_std_name"]

#             if not name2 or not names_agree(name1, name2):
#                 continue

#             score = JaroWinkler.similarity(name1, name2)

#             if score >= threshold and score > best_score:
#                 best_score = score
#                 best_match = {
#                     "candidate_2014": row1[name_col],
#                     "candidate_2019": row2[name_col],
#                     "party_2014": row1[party_col],
#                     "party_2019": row2[party_col],
#                     "similarity": score,
#                     "index_2014": i,
#                     "index_2019": j
#                 }

#         if best_match is not None:
#             results.append(best_match)

#     return pd.DataFrame(results)
# matching_gujarat_14_19 = match_repeat_candidates(
#     gujarat_14,
#     gujarat_19,
#     threshold=0.70
# )

# matching_gujarat_14_19


,candidate_2014,candidate_2019,party_2014,party_2019,similarity,index_2014,index_2019
0,Babubhai Chhaganbhai Talaviya,Babubhai Talaviya,Bahujan Mukti Party,Bahujan Mukti Party,0.881947,10,15
1,Bathwar Nanjibhai Amarshibhai,Bathvar Nanji Amarshi,IND,IND,0.898407,16,20
2,Bhabhor Jasvantsinh Sumanbhai,Jasvantsinh Sumanbhai Bhabhor,BJP,BJP,0.732280,17,129
3,Bhoraniya Soyabbhai Hasambhai,Bhoraniya Soyabbhai Hasambhai,IND,IND,1.000000,22,39
4,Chandravijaysinh Takhubha Rana,Chandansinh,IND,IND,0.828485,31,44
5,Chauhan Devusinh Jesingbhai,Chauhan Devusinh,BJP,BJP,0.918519,35,54
6,Chavada Ashokbhai Nathabhai,Chavada Ashok,IND,IND,0.896296,38,64
7,Chavda Palabhai Nanjibhai,Chavada Shamjibhai,IND,IND,0.830778,42,65
8,Chavda Vinod Lakhmashi,Chavda Vinod Lakhamshi,BJP,BJP,0.990909,43,69
9,Chudasama Rajeshbhai Naranbhai,Chudasama Rajeshbhai Naranbhai,BJP,BJP,1.000000,45,72


**Adding new columns to data**

In [24]:
def process_election_data(state, year, matching_df):
    file_path = f"cleaned_data/imputed_parquet/{state}_{year}.parquet"
    df = pd.read_parquet(file_path)
    
    df["winner"] = False
    df["constituency"] = ""
    
    mapping_df = matching_df[['election_candidate', 'winner_of_constituency']].copy()
    mapping_df.columns = ['Candidate', 'matched_constituency']
    
    df = df.merge(mapping_df, on='Candidate', how='left')
    
    is_winner = df['matched_constituency'].notna()
    df.loc[is_winner, 'winner'] = True
    df.loc[is_winner, 'constituency'] = df.loc[is_winner, 'matched_constituency']
    
    df.drop(columns=['matched_constituency'], inplace=True)
    return df

In [25]:
datasets = [
    ("andhra", 2014, matching_andhra_14),
    ("andhra", 2019, matching_andhra_19),
    ("andhra", 2024, matching_andhra_24),
    ("bihar", 2014, matching_bihar_14),
    ("bihar", 2019, matching_bihar_19),
    ("bihar", 2024, matching_bihar_24),
    ("gujarat", 2014, matching_gujarat_14),
    ("gujarat", 2019, matching_gujarat_19),
    ("gujarat", 2024, matching_gujarat_24),
]